# Full Experiment Suite - Google Colab

Run the dissertation forecasting pipeline on Google Colab and export the generated experiment data. The notebook calls the repository CLI; it does not reimplement ingestion, preprocessing, splitting, scaling, training, or evaluation logic.

Use `configs/default.yaml` for the full dissertation experiment. Use `configs/smoke.yaml` only for a quick software check.

## 1. Get the repository

If this notebook is opened from a cloned repository, the cell uses the current checkout. If it is opened directly in Colab, set `REPO_URL` before running the cell.

In [ ]:
from pathlib import Path
import datetime as dt
import json
import os
import shutil
import subprocess
import sys

IN_COLAB = Path("/content").exists()
REPO_URL = ""  # Example: "https://github.com/<owner>/local-weather-forecasting-ml.git"
BRANCH = "main"
REPO_DIR = Path("/content/local-weather-forecasting-ml") if IN_COLAB else Path.cwd()

def run(command, cwd=None):
    command = [str(part) for part in command]
    print("+", " ".join(command))
    return subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)

if IN_COLAB and not (REPO_DIR / "pyproject.toml").exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL to this repository's Git URL, then rerun this cell.")
    run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())

## 2. Install dependencies

Colab already provides PyTorch on many runtimes, but installing from `requirements.txt` keeps the environment aligned with the project.

In [ ]:
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
run([sys.executable, "-m", "pip", "install", "-e", "."])

## 3. Add Weathercloud CSV files

Choose one source for the raw station exports. Keep the inputs observation-only: do not add NWP outputs, external weather forecasts, or other-station data.

In [ ]:
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Option A: upload CSV files through the Colab file picker.
UPLOAD_FROM_BROWSER = True

# Option B: copy CSV files from Google Drive. Set USE_DRIVE_DATA=True and mount Drive first.
USE_DRIVE_DATA = False
DRIVE_RAW_DIR = Path("/content/drive/MyDrive/weathercloud_raw")

if USE_DRIVE_DATA:
    if IN_COLAB and not Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    if not DRIVE_RAW_DIR.exists():
        raise FileNotFoundError(f"Drive raw-data folder not found: {DRIVE_RAW_DIR}")
    for source in sorted(DRIVE_RAW_DIR.glob("*.csv")):
        target = RAW_DIR / source.name
        shutil.copy2(source, target)
        print("Copied", source, "->", target)
elif UPLOAD_FROM_BROWSER:
    try:
        from google.colab import files
    except ModuleNotFoundError:
        print("Not running in Colab. Place Weathercloud CSV files in data/raw manually.")
    else:
        uploaded = files.upload()
        for name, data in uploaded.items():
            target = RAW_DIR / Path(name).name
            target.write_bytes(data)
            print("Uploaded", target)

raw_files = sorted(RAW_DIR.glob("*.csv"))
if not raw_files:
    raise FileNotFoundError("No Weathercloud CSV files found in data/raw.")

print(f"Raw CSV files ready: {len(raw_files)}")
for path in raw_files:
    print("-", path, f"({path.stat().st_size / 1024 / 1024:.1f} MiB)")

Optional: mount Google Drive manually. The raw-data and export cells can also mount Drive automatically when their Drive toggles are enabled.

In [ ]:
MOUNT_DRIVE = False

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 4. Run the experiment

`RUN_SMOKE_FIRST=True` is useful for checking the Colab runtime before spending time on the full configuration. The full experiment uses `configs/default.yaml`.

In [ ]:
CONFIG_PATH = "configs/default.yaml"
FRESH_RUN = True
RUN_SMOKE_FIRST = False

if RUN_SMOKE_FIRST:
    run([sys.executable, "-m", "weather_forecasting_pipeline", "run-all", "--config", "configs/smoke.yaml", "--fresh"])

command = [sys.executable, "-m", "weather_forecasting_pipeline", "run-all", "--config", CONFIG_PATH]
if FRESH_RUN:
    command.append("--fresh")
run(command)

## 5. Inspect key outputs

These checks confirm that the notebook produced dissertation-style outputs rather than smoke-test outputs.

In [ ]:
import pandas as pd

metrics_path = Path("artifacts/metrics/metrics.csv")
summary_path = Path("artifacts/reports/summary.md")

if not metrics_path.exists():
    raise FileNotFoundError(metrics_path)

metrics = pd.read_csv(metrics_path)
display(metrics.sort_values(["horizon_steps", "rmse"]).head(30))

print("Project names:", sorted(metrics["project"].dropna().unique()) if "project" in metrics.columns else "project column absent")
print("Horizons:", sorted(metrics["horizon_label"].dropna().unique()))
print("Models:", sorted(metrics["model"].dropna().unique()))

if summary_path.exists():
    print("\nSummary preview:\n")
    print("\n".join(summary_path.read_text(encoding="utf-8").splitlines()[:40]))

## 6. Snapshot the run

The snapshot freezes configs, generated datasets, predictions, metrics, plots, reports, models, and scalers into `runs/<run_id>/`. For smaller exports, set the skip flags to `True` before running the cell.

In [ ]:
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%y%m%d_%H%M_colab")
SKIP_SUPERVISED = False
SKIP_INTERIM = False
NO_PLOTS = False

snapshot_command = [sys.executable, "scripts/snapshot_run.py", "--run-id", RUN_ID, "--force"]
if SKIP_SUPERVISED:
    snapshot_command.append("--skip-supervised")
if SKIP_INTERIM:
    snapshot_command.append("--skip-interim")
if NO_PLOTS:
    snapshot_command.append("--no-plots")

run(snapshot_command)
SNAPSHOT_DIR = Path("runs") / RUN_ID
print("Snapshot:", SNAPSHOT_DIR.resolve())

## 7. Export the generated data

The archive contains the run snapshot. In Colab, downloading very large archives through the browser can fail; copying to Drive is usually more reliable.

In [ ]:
EXPORT_DIR = Path("exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

archive_path = shutil.make_archive(
    str(EXPORT_DIR / RUN_ID),
    "zip",
    root_dir=SNAPSHOT_DIR.parent,
    base_dir=SNAPSHOT_DIR.name,
)
archive_path = Path(archive_path)
print("Archive:", archive_path.resolve())
print(f"Size: {archive_path.stat().st_size / 1024 / 1024:.1f} MiB")

COPY_TO_DRIVE = False
DRIVE_EXPORT_DIR = Path("/content/drive/MyDrive/weather_forecasting_runs")
if COPY_TO_DRIVE:
    if IN_COLAB and not Path("/content/drive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    drive_target = DRIVE_EXPORT_DIR / archive_path.name
    shutil.copy2(archive_path, drive_target)
    print("Copied archive to", drive_target)

DOWNLOAD_FROM_BROWSER = False
if DOWNLOAD_FROM_BROWSER:
    try:
        from google.colab import files
    except ModuleNotFoundError:
        print("Browser download is only available in Colab.")
    else:
        files.download(str(archive_path))

## 8. Export manifest

Use this cell to list the most important exported files after the run.

In [ ]:
important_paths = [
    Path("data/interim/canonical.parquet"),
    Path("data/interim/prepared.parquet"),
    Path("artifacts/metrics/metrics.csv"),
    Path("artifacts/reports/summary.md"),
    SNAPSHOT_DIR / "manifest.json",
    archive_path,
]

for path in important_paths:
    if path.exists():
        print(f"{path} - {path.stat().st_size / 1024 / 1024:.2f} MiB")
    else:
        print(f"missing: {path}")

manifest_path = SNAPSHOT_DIR / "manifest.json"
if manifest_path.exists():
    print("\nSnapshot manifest:\n")
    print(json.dumps(json.loads(manifest_path.read_text(encoding="utf-8")), indent=2)[:4000])